# Preprocessor Pipeline
### Golden Plains Roadside Biodiversity

Here is a little bit of information on the data you are given. Golden Plains Shire (Australia) is responsible for managing 1834 kilometres of road reserves. Road reserves are not only used for transport but also act as service corridors, in fire prevention, recreation, and occasionally agricultural pursuits. Native vegetation on roadsides is important flora and fauna habitat and landscape character.

In 2014, Golden Plains Shire acquired funding through the Victorian Adaptation and Sustainability Partnership (VASP) to undertake Councils ‘Building Adaptive Capacity on Roadsides’ project. The Project was designed to identify significant environmental assets on roadsides, improve roadside management practices and reduce Council’s risk of potential breaches against Federal and State environmental legislation. 

The council made this <a href='https://data.gov.au/data/dataset/golden-plains-roadside-biodiversity'>dataset available here</a>.<br>
![plain](https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Mount_Conner%2C_August_2003.jpg/375px-Mount_Conner%2C_August_2003.jpg)
<br>

🎯 Today, you will work with a simplified version of this real dataset. The dataset contains a number of biodiversity observations including one on tree size (`RCACTreesS`)

🎯 As last time, the exercise will consist of the data preparation and feature selection techniques you have learnt: we will again try to predict via linear regression the `RCACScore`. But this time, we will do it using `Pipelines`, and we don't really care that much about the final score (the goal is to demonstrate how pipelines work.

The goal is to demonstrate how helpful pipelines are in making our code cleaner, and avoid repetition.

👇 Load the data into this notebook as a pandas dataframe named `df`, and display its first 5 rows.

Use `../../Excercise_data/L04/biodiversity.csv`.

# Remove Duplicates and Hold Out

Remove accidental duplicate records before splitting. Identical legitimate observations are not automatically data errors. Why? Because if you don't do it now, once you train-test-split (hold out method) you might have data leakage. Do the following:
1. Remove the duplicates
2. Create the target `y` vector (`RCACScore`) and feature `X` matrix
3. Using `random_state=42` and a 30% split for the test set, create the `X_train`, `X_test`, `y_train`, `y_test`
4. divide the Categorical and numerical features
Exclude `SiteCode` from `X`: it identifies the surveyed site rather than measuring its ecology. Keep the other features so the two pipelines compare the same prediction problem.


# Creating our numeric pipeline

Let's start by taking care of the numeric values, and do this in a pipeline. 
1. Impute values with a `SimpleImputer`, with the default strategy.
2. Scale the values using a `StandardScaler`.

Create a pipeline that does just that! Call it `num_pipe`. 

# Categorical Data Pipeline

Let's now build a new `pipeline` and name it `cat_pipe` for **Categorical Pipeline**. We will deal with the categorical data by doing the following:
- Apply a `SimpleImputer()` with the `most_frequent` strategy
- Apply a `OneHotEncoder()` with the `handle_unknown="ignore"` and `sparse_output=True` to the categorical data


Keep one-hot output sparse for this dataset. Dense unregularised least squares can be numerically unstable with many redundant indicator columns.


# Building a single processing Pipeline

Ok, the process above was already saving us a lot of time. But now, we are going to take this to the next level by creating a `ColumnTransformer` that groups both `pipelines`together! Name your `ColumnTransformer` as `preprocessor` (**preprocessing pipeline**). If in doubt, consult the documentation!


Set `sparse_threshold=1.0` in both ColumnTransformers so sparse encoding is retained when combined with the numeric branch. This lets `LinearRegression` use its sparse least-squares solver.


# Applying your pipline
you can either fit your num_pipe and cat_pipe on your numeric_feature and categorical_features respectively or a better and easier way is to
fit_transform your `preprocessor` combined pipline on your X_train and transform it on the X_test

# Treating Different Numeric Features Differently in a Pipeline
Looking at the distribution of the `X_train[numerical_features]`:
we can define different scalers:

- standard_scaler_features = ['PowerlineD', 'LengthM', 'RCACGround', 'RCACLogs']
- minmax_scaler_features = ['RoadWidthM', 'EVCCode', 'BiomassRed', 'RCACWeedCo', 'Weeds']
- robust_scaler_features = ['Trees', 'RCACWetlan', 'HollowTree', 'LogsWoodyD', 'RCACRareSp']

Creat a new preprocessed pipline considering above and call the new preprocessor pipline: `preprocessor_2` so we can at the end test both of the pipelines on the model


These groups demonstrate different transformers; distribution shape does not by itself require a particular scaler. Keep other numeric columns using the original numeric pipeline rather than silently dropping them. Compare complete preprocessing-plus-model pipelines using training-only cross-validation.

## Apply both preprocessors with Linear Regression
Use the same training cross-validation folds to compare the two complete pipelines, with preprocessing fitted inside each fold. Select using mean validation RMSE. Affine rescaling alone should not materially change unregularised least-squares predictions when the same features are retained; numerical differences may remain.

## Report the held-out test performance
Refit the selected pipeline on all training data, then evaluate the test set once. Do not select the pipeline using its test score.